# One-fold experiment

Change the settings in the next code cell and run the notebook from the top. It trains one fold with the same runner as `python -m src.experiment`: criteria network, overall network, single-rating network, and the five classical baselines.

The full five-fold run is appended to `results/experiment.log`. This notebook appends its own epochs to `results/notebook.log`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Settings

In [2]:
FOLD = 0
MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 512
TOP_K = 10
SEED = 42


## Train this fold

Neural nets use the fit split and stop early on the validation split. Baselines train on fit and validation together. Test ratings are the held-out 20%.

In [3]:
import torch
from src.data import CRITERIA, load_ratings, make_folds
from src.experiment import run_fold
from src.log import configure

configure(ROOT / 'results' / 'notebook.log')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ratings = load_ratings(ROOT / 'texts.rar')
fold = make_folds(len(ratings), n_folds=5, seed=SEED)[FOLD]
print(
    f'device {device}  fold {FOLD}  '
    f'fit {len(fold["fit"])}  val {len(fold["val"])}  test {len(fold["test"])}'
)
result = run_fold(
    ratings,
    fold,
    FOLD,
    device=device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    top_k=TOP_K,
    seed=SEED,
)


2026-10-12 02:52:26  logging to /run/media/advaitam/OS/Users/advay/Desktop/coding-stuff/rds-paper-implementation/results/notebook.log


2026-10-12 02:52:33  fold 0: criteria DNN


device cuda  fold 0  fit 64057  val 811  test 16217


2026-10-12 02:52:35    epoch 01  val_mae 0.7981


2026-10-12 02:52:35    epoch 02  val_mae 0.7935


...

2026-10-12 02:53:00  fold 0: baseline


2026-10-12 02:53:02  fold 0: knn


2026-10-12 02:53:04  fold 0: slope_one


2026-10-12 02:53:05  fold 0: svd


2026-10-12 02:53:19  fold 0: svdpp


## This fold

In [4]:
print(f"{'aspect':<24}MAE")
labels = {
    'value': 'Value',
    'rooms': 'Rooms',
    'location': 'Location',
    'cleanliness': 'Cleanliness',
    'checkin': 'Check in / front desk',
    'service': 'Service',
    'business': 'Business service',
    'mean': 'Mean',
}
for name in list(CRITERIA) + ['mean']:
    print(f"{labels[name]:<24}{result['criteria_mae'][name]:.4f}")
print()
print(f"{'method':<20} {'MAE':>8} {'FCP':>8}")
order = [
    ('ours', 'Our'),
    ('single_dnn', 'Single DNN'),
    ('svd', 'SVD'),
    ('svdpp', 'SVD++'),
    ('slope_one', 'Slope One'),
    ('baseline', 'Baseline'),
    ('knn', 'KNN with baseline'),
]
for key, label in order:
    row = result['methods'][key]
    print(f"{label:<20} {row['mae']:8.4f} {row['fcp']:8.4f}")


aspect                  MAE
Value                   0.8389
Rooms                   0.7930
Location                0.6866
Cleanliness             0.6737
Check in / front desk   0.7955
Service                 0.7860
Business service        0.9003
Mean                    0.7820

method                    MAE      FCP
Our                    0.7624   0.6935
Single DNN             0.7889   0.6726
SVD                    0.8108   0.6726
SVD++                  0.8257   0.6786
Slope One              0.8971   0.6071
Baseline               0.8139   0.6726
KNN with baseline      0.8148   0.6696
